# Phase 2c (Exploratory) — Per-Gene Alteration State Model & Mechanism-Specific Co-alteration

**Status: exploratory, not folded into the main pipeline.** Follows on from
`05_phase2b_snv_cnv_exploration.ipynb`.

The idea being tested: instead of collapsing everything into a binary
`altered = 0/1`, annotate **each gene in each sample** with *what kind* of
alteration it carries:

| state | meaning |
|---|---|
| 0 | tested, no alteration |
| 1 | SNV/indel only |
| 2 | deep CNV only (deep amp / deep del) |
| 3 | **both** SNV and deep CNV in the same gene, same sample |

The motivation is that the binary matrix can't tell a mono-allelic hit apart
from a fully inactivated gene, so gene-pair results lose that layer of
biology.

**What this notebook concludes** (details in Section 6):

1. The state model is worth building, but **state 3 is not a two-hit marker** --
   only 22% of state-3 events are the biologically meaningful case, and it is
   a *different* event class from the Phase 1 two-hit feature (no overlap at all).
2. A full 4x4 categorical contingency test is **too sparse to trust** -- shown
   concretely in Section 4, worst cells hold 1-5 samples.
3. **Mechanism-specific 2x2 tests are the version of this idea that works**
   (Section 5). They stay statistically robust, and they find real biology that
   *no* single-mechanism matrix can see -- e.g. `MDM2` amplification vs `TP53`
   mutation exclusivity (OR=0.05), which needs one gene's CNV compared against
   the other gene's SNV.
4. The driver-consistency filter from notebook 05 **demonstrably removes
   amplicon-passenger artifacts** (Section 5b) while leaving textbook biology
   intact.


In [1]:
import itertools, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("future.no_silent_downcasting", True)

DATA_DIR = Path.cwd().parent / "data"
PROCESSED_DIR, EXTERNAL_DIR = DATA_DIR / "processed", DATA_DIR / "external"

alterations = pd.read_parquet(PROCESSED_DIR / "alterations_long.parquet")
panel_coverage = pd.read_parquet(PROCESSED_DIR / "panel_gene_coverage.parquet")
clinical = pd.read_parquet(PROCESSED_DIR / "clinical_tidy.parquet")
genes_by_ct = (pd.read_parquet(PROCESSED_DIR / "consensus_genes_per_cancer_type.parquet")
                 .groupby("CANCER_TYPE")["Hugo_Symbol"].apply(set).to_dict())
biallelic_p1 = pd.read_parquet(PROCESSED_DIR / "putative_biallelic_tsg_events.parquet")
with open(EXTERNAL_DIR / "oncokb_curated_genes.json") as f:
    GENE_ROLE = {g["hugoSymbol"]: g["geneType"] for g in json.load(f)}

samples = clinical[["SAMPLE_ID", "SEQ_ASSAY_ID", "CANCER_TYPE"]].drop_duplicates("SAMPLE_ID")
print(alterations.shape, len(samples))

(1974322, 13) 271837


## 1. Prerequisite: which panels can a CNV state even come from?

Notebook 05 found that the `cna_capable` flag (self-reported panel metadata)
is not trustworthy -- 21 of 48 flagged panels report zero deep CNA calls ever.
That matters doubly here: on those panels a gene can **never** be assigned
state 2 or 3, so leaving them in silently deflates both states. Everything
below uses the empirically verified panel set.

In [2]:
cnv_sample_set = set(alterations.loc[alterations["Alteration_Type"] == "CNV", "Sample_ID"])
panel_yield = (samples.assign(has_cnv=samples["SAMPLE_ID"].isin(cnv_sample_set))
                      .groupby("SEQ_ASSAY_ID").agg(n=("SAMPLE_ID", "nunique"), n_cnv=("has_cnv", "sum")))
panel_yield["frac"] = panel_yield["n_cnv"] / panel_yield["n"]

claimed_capable = set(panel_coverage.loc[panel_coverage["cna_capable"], "SEQ_ASSAY_ID"])
CNV_OK_PANELS = set(panel_yield[panel_yield["frac"] >= 0.01].index) & claimed_capable
cnv_ok_samples = set(samples.loc[samples["SEQ_ASSAY_ID"].isin(CNV_OK_PANELS), "SAMPLE_ID"])

print(f"panels claiming cna_capable : {len(claimed_capable)}")
print(f"panels actually reporting   : {len(CNV_OK_PANELS)}")
print(f"samples where a CNV state is even possible: {len(cnv_ok_samples):,} of {len(samples):,} "
      f"({len(cnv_ok_samples)/len(samples):.1%})")

panels claiming cna_capable : 48
panels actually reporting   : 27
samples where a CNV state is even possible: 179,513 of 271,837 (66.0%)


## 2. The state model

One row per (sample, gene) that carries any alteration. State 0 stays implicit
(tested but clean) -- materialising it would mean ~50M rows for no gain.

In [3]:
alt_ok = alterations[alterations["Sample_ID"].isin(cnv_ok_samples)]

flags = (alt_ok.assign(is_snv=alt_ok["Alteration_Type"].eq("MUT"),
                       is_cnv=alt_ok["Alteration_Type"].eq("CNV"))
               .groupby(["Sample_ID", "Hugo_Symbol"], observed=True)
               .agg(has_snv=("is_snv", "any"), has_cnv=("is_cnv", "any")))

cna_dir = (alt_ok[alt_ok["Alteration_Type"] == "CNV"]
           .drop_duplicates(["Sample_ID", "Hugo_Symbol"])
           .set_index(["Sample_ID", "Hugo_Symbol"])["CNA_Direction"])

states = flags.join(cna_dir).reset_index()
states["state"] = np.where(states["has_snv"] & states["has_cnv"], 3,
                    np.where(states["has_snv"], 1, 2))
states["Gene_Role"] = states["Hugo_Symbol"].map(GENE_ROLE)

labels = {1: "1 = SNV only", 2: "2 = CNV only", 3: "3 = SNV + CNV (same gene)"}
print(states["state"].value_counts().sort_index().rename(labels))
print((states["state"].value_counts(normalize=True).sort_index() * 100).round(2).astype(str) + "%")
print(f"\nstate-3 events: {int((states['state']==3).sum()):,} across "
      f"{states.loc[states['state']==3,'Sample_ID'].nunique():,} samples and "
      f"{states.loc[states['state']==3,'Hugo_Symbol'].nunique()} genes")

state
1 = SNV only                 844929
2 = CNV only                 362816
3 = SNV + CNV (same gene)      8498
Name: count, dtype: int64
state
1    69.47%
2    29.83%
3      0.7%
Name: proportion, dtype: object

state-3 events: 8,498 across 7,878 samples and 555 genes


**State 3 is rare: 0.7% of altered gene-sample pairs.** (It looks like
0.49% if the CNV-blind panels are wrongly left in the denominator.) That
number drives most of what follows -- it is why the 4x4 categorical test in
Section 4 cannot hold up, and why Section 5 uses a different formulation.

## 3. Is state 3 the same thing as a two-hit?

**No, and this is the important correction.** Two separate reasons:

**(a) Direction matters.** A state-3 event is only two-hit-like when the gene
is a tumour suppressor *and* the CNV is a deletion. An oncogene that is both
mutated and amplified is two driver events stacking up, not biallelic
inactivation.

**(b) Deep deletion is already biallelic.** The classic Knudson two-hit is
*shallow* deletion (one copy lost) + a mutation on the surviving copy. A
**deep** deletion means both copies are gone -- there is no remaining allele
for a mutation to sit on. So `TSG + deep deletion + SNV` is not the clean
Knudson case either; it more likely reflects subclonal mixture (part of the
tumour deleted, part mutated) or an imprecise call.

Phase 1's `putative_biallelic_tsg_events.parquet` already captures the real
Knudson case (shallow del + damaging mutation). The two are disjoint by
construction -- verified below.

In [4]:
s3 = states[states["state"] == 3].copy()

def classify_state3(row):
    role, direction = row["Gene_Role"], row["CNA_Direction"]
    if role in ("TSG", "ONCOGENE_AND_TSG") and direction == "Deep_Deletion":
        return "TSG + deep deletion (closest to two-hit, but see caveat b)"
    if role in ("ONCOGENE", "ONCOGENE_AND_TSG") and direction == "Amplification":
        return "Oncogene + amplification (two driver events, NOT two-hit)"
    if role == "TSG" and direction == "Amplification":
        return "TSG + amplification (direction-inconsistent)"
    if role == "ONCOGENE" and direction == "Deep_Deletion":
        return "Oncogene + deep deletion (direction-inconsistent)"
    return "Unannotated gene role"

s3["s3_class"] = s3.apply(classify_state3, axis=1)
print(s3["s3_class"].value_counts())
print((s3["s3_class"].value_counts(normalize=True) * 100).round(1).astype(str) + "%")
print("\nTop 10 genes driving state 3:")
print(s3.groupby(["Hugo_Symbol", "Gene_Role"]).size().sort_values(ascending=False).head(10))

s3_class
Oncogene + amplification (two driver events, NOT two-hit)     4977
TSG + deep deletion (closest to two-hit, but see caveat b)    1889
TSG + amplification (direction-inconsistent)                   942
Unannotated gene role                                          578
Oncogene + deep deletion (direction-inconsistent)              112
Name: count, dtype: int64
s3_class
Oncogene + amplification (two driver events, NOT two-hit)     58.6%
TSG + deep deletion (closest to two-hit, but see caveat b)    22.2%
TSG + amplification (direction-inconsistent)                  11.1%
Unannotated gene role                                          6.8%
Oncogene + deep deletion (direction-inconsistent)              1.3%
Name: proportion, dtype: object

Top 10 genes driving state 3:
Hugo_Symbol  Gene_Role
EGFR         ONCOGENE     1645
TP53         TSG           696
KRAS         ONCOGENE      453
ERBB2        ONCOGENE      397
PIK3CA       ONCOGENE      358
PDGFRA       ONCOGENE      173
CDKN2A   

In [5]:
p1_events = set(map(tuple, biallelic_p1[["Sample_ID", "Hugo_Symbol"]].drop_duplicates().values))
s3_tsg_deep = set(map(tuple, s3.loc[s3["s3_class"].str.startswith("TSG + deep"),
                                    ["Sample_ID", "Hugo_Symbol"]].values))
print(f"Phase 1 two-hit  (shallow del + damaging mut) : {len(p1_events):,} events")
print(f"State-3 TSG + deep deletion (this notebook)   : {len(s3_tsg_deep):,} events")
print(f"overlap                                       : {len(p1_events & s3_tsg_deep):,}")
print("\n-> disjoint, as expected: a gene call is either -1 or -2, never both.")
print("   These are two DIFFERENT event classes, not two versions of the same one.")

Phase 1 two-hit  (shallow del + damaging mut) : 23,632 events
State-3 TSG + deep deletion (this notebook)   : 1,889 events
overlap                                       : 0

-> disjoint, as expected: a gene call is either -1 or -2, never both.
   These are two DIFFERENT event classes, not two versions of the same one.


So of 8,498 state-3 events, only ~22% (1,897) are even
direction-plausible as inactivation, and those are *still* not the Knudson
case Phase 1 already models. `EGFR` alone contributes ~1,650 state-3 events
(mutation + amplification in lung cancer / glioma) -- a real and well-known
dual-driver pattern, but the opposite of a two-hit.

**Conclusion:** keep state 3 as a descriptive annotation of "this gene got hit
two different ways". Do not label it two-hit.

## 4. Why the full 4x4 categorical test doesn't hold up

The natural next step is a 4x4 contingency table (gene A state x gene B state).
Building one for a deliberately favourable case -- `EGFR` x `TP53` in NSCLC,
two of the most frequently altered genes in the largest cancer type:

In [6]:
ct = "Non-Small Cell Lung Cancer"
ct_samples = samples[(samples["CANCER_TYPE"] == ct) & samples["SEQ_ASSAY_ID"].isin(CNV_OK_PANELS)].set_index("SAMPLE_ID")

def state_vector(gene):
    sub = alt_ok[(alt_ok["Hugo_Symbol"] == gene) & alt_ok["Sample_ID"].isin(ct_samples.index)]
    g = (sub.assign(s=sub["Alteration_Type"].eq("MUT"), c=sub["Alteration_Type"].eq("CNV"))
            .groupby("Sample_ID").agg(a=("s", "any"), b=("c", "any")))
    v = pd.Series(0, index=ct_samples.index)
    v.loc[g.index[g["a"] & g["b"]]] = 3
    v.loc[g.index[g["a"] & ~g["b"]]] = 1
    v.loc[g.index[~g["a"] & g["b"]]] = 2
    return v

cov = panel_coverage[panel_coverage["Hugo_Symbol"].isin(["EGFR", "TP53"])].groupby("SEQ_ASSAY_ID")["Hugo_Symbol"].apply(set)
both_covered = cov[cov.apply(lambda s: {"EGFR", "TP53"} <= s)].index
eligible = ct_samples["SEQ_ASSAY_ID"].isin(both_covered)

table = pd.crosstab(state_vector("EGFR")[eligible], state_vector("TP53")[eligible])
table.index.name, table.columns.name = "EGFR state", "TP53 state"
print(table)
print(f"\ncells with < 5 samples: {(table.values < 5).sum()} of {table.size}; smallest = {table.values.min()}")

TP53 state     0     1   2   3
EGFR state                    
0           8695  8006  37  13
1           2121  2389  13   4
2             96   235   1   2
3            177   718   5   5

cells with < 5 samples: 3 of 16; smallest = 1


Even in the single most favourable pair available, the state-3 row and
column collapse to cells of **1, 2, 4 and 5 samples**. Any omnibus test on
this table is carried by those cells, and an omnibus p-value doesn't even tell
you *which* combination drove it or in which direction. For a typical pair in
a mid-sized cancer type it is far worse.

**So the 4x4 model is a dead end as a primary test** -- not because the idea
is wrong, but because state 3 is too rare to support it. The next section
keeps the same idea in a form that survives.

## 5. Mechanism-specific 2x2 tests -- the version that works

Instead of one 4x4 table per pair, run **four separate 2x2 tests** per pair:

| test | question |
|---|---|
| A-SNV x B-SNV | do point mutations in A and B co-occur? |
| A-SNV x B-CNV | does mutating A go with copy-number change in B? |
| A-CNV x B-SNV | the reverse (asymmetric -- both are needed) |
| A-CNV x B-CNV | do the copy-number events co-occur? |

Each stays a clean 2x2 with a real odds ratio and direction, each gets its own
BH-FDR family, and together they answer exactly what the categorical model was
after: *which mechanism combination carries this pair's signal*.

All four use a **common denominator** (the CNV-reporting subset of that cancer
type), so the four results are directly comparable to each other -- otherwise
the SNV-SNV test would be computed on a larger population than the rest and
the comparison would be meaningless.

In [7]:
PILOT = ["Breast Cancer", "Non-Small Cell Lung Cancer", "Bladder Cancer"]
MECHS = [("SNV", "SNV"), ("SNV", "CNV"), ("CNV", "SNV"), ("CNV", "CNV")]

CONSISTENT = {"TSG": {"Deep_Deletion"}, "ONCOGENE": {"Amplification"},
              "ONCOGENE_AND_TSG": {"Deep_Deletion", "Amplification"}}
cnv_rows = alterations[alterations["Alteration_Type"] == "CNV"].copy()
cnv_rows["role"] = cnv_rows["Hugo_Symbol"].map(GENE_ROLE)
cnv_rows["driver_ok"] = [r in CONSISTENT and d in CONSISTENT[r]
                         for r, d in zip(cnv_rows["role"], cnv_rows["CNA_Direction"])]

snv_pairs = alterations.loc[alterations["Alteration_Type"] == "MUT", ["Sample_ID", "Hugo_Symbol"]].drop_duplicates()
cnv_pairs_all = cnv_rows[["Sample_ID", "Hugo_Symbol"]].drop_duplicates()
cnv_pairs_drv = cnv_rows.loc[cnv_rows["driver_ok"], ["Sample_ID", "Hugo_Symbol"]].drop_duplicates()


def bh_fdr(p):
    p = np.asarray(p); n = len(p); o = np.argsort(p)
    r = np.empty(n, int); r[o] = np.arange(1, n + 1)
    q = p * n / r
    qs = np.minimum.accumulate(q[o][::-1])[::-1]
    out = np.empty(n); out[o] = np.clip(qs, 0, 1)
    return out


def run_mechanism_tests(cnv_source, label):
    frames = []
    for ct in PILOT:
        cs = samples[(samples["CANCER_TYPE"] == ct) & samples["SEQ_ASSAY_ID"].isin(CNV_OK_PANELS)].set_index("SAMPLE_ID")
        n = len(cs)
        sub = alterations.loc[alterations["CANCER_TYPE"] == ct, ["Sample_ID", "Hugo_Symbol"]].drop_duplicates()
        cnt = sub[sub["Sample_ID"].isin(cs.index)].groupby("Hugo_Symbol")["Sample_ID"].nunique()
        genes = sorted(set(cnt[cnt >= max(5, 0.03 * n)].index) & genes_by_ct.get(ct, set()))
        if len(genes) < 2:
            continue

        def mat(src):
            s = src[src["Sample_ID"].isin(cs.index) & src["Hugo_Symbol"].isin(genes)]
            return (s.assign(v=True).pivot(index="Sample_ID", columns="Hugo_Symbol", values="v")
                     .reindex(index=cs.index, columns=genes).fillna(False).astype(bool))

        M_snv, M_cnv = mat(snv_pairs), mat(cnv_source)
        pm = (panel_coverage[panel_coverage["Hugo_Symbol"].isin(genes)].assign(c=True)
              .pivot(index="SEQ_ASSAY_ID", columns="Hugo_Symbol", values="c").reindex(columns=genes).fillna(False))
        coverage = pm.reindex(cs["SEQ_ASSAY_ID"]).fillna(False).astype(bool)
        coverage.index = cs.index

        rows = []
        for a, b in itertools.combinations(genes, 2):
            elig = (coverage[a] & coverage[b]).to_numpy()
            if elig.sum() < 50:
                continue
            for ma, mb in MECHS:
                A = (M_snv[a] if ma == "SNV" else M_cnv[a]).to_numpy()[elig]
                B = (M_snv[b] if mb == "SNV" else M_cnv[b]).to_numpy()[elig]
                both = int((A & B).sum())
                if both < 5:
                    continue
                a_only, b_only = int((A & ~B).sum()), int((~A & B).sum())
                neither = int((~A & ~B).sum())
                orv, pv = fisher_exact([[both, a_only], [b_only, neither]])
                rows.append(dict(Cancer_Type=ct, Gene_A=a, Gene_B=b, mech_A=ma, mech_B=mb,
                                 n_elig=int(elig.sum()), n_both=both, n_A_only=a_only,
                                 n_B_only=b_only, n_neither=neither, odds_ratio=orv, p_value=pv))
        if rows:
            df = pd.DataFrame(rows)
            df["q_value"] = np.nan
            for _, g in df.groupby(["mech_A", "mech_B"]):       # own BH family per combo
                df.loc[g.index, "q_value"] = bh_fdr(g["p_value"].to_numpy())
            frames.append(df)

    res = pd.concat(frames, ignore_index=True)
    res["combo"] = res["mech_A"] + "-" + res["mech_B"]
    print(f"[{label}] tested={len(res):,}  significant={int((res['q_value']<0.05).sum()):,}")
    print(res.groupby("combo").agg(tested=("p_value", "size"),
                                   significant=("q_value", lambda s: int((s < 0.05).sum()))))
    return res


mech_all = run_mechanism_tests(cnv_pairs_all, "all deep CNVs")
print()
mech_drv = run_mechanism_tests(cnv_pairs_drv, "driver-consistent CNVs only")

[all deep CNVs] tested=4,876  significant=2,166
         tested  significant
combo                       
CNV-CNV     754          475
CNV-SNV     953          236
SNV-CNV     953          138
SNV-SNV    2216         1317



[driver-consistent CNVs only] tested=4,274  significant=1,926
         tested  significant
combo                       
CNV-CNV     487          293
CNV-SNV     793          199
SNV-CNV     778          117
SNV-SNV    2216         1317


### 5a. What the cross-mechanism tests find that no single matrix can

The `SNV x CNV` combinations are the whole point: they compare one gene's
point mutations against the *other* gene's copy-number state. An SNV-only
matrix cannot see these, and neither can a CNV-only matrix.

In [8]:
sig = mech_all[mech_all["q_value"] < 0.05]
cross = sig[sig["mech_A"] != sig["mech_B"]]
print(f"significant cross-mechanism results: {len(cross)}\n")
print("Strongest mutual exclusivity (cross-mechanism):")
print(cross[cross["odds_ratio"] < 1].nsmallest(8, "odds_ratio")[
    ["Cancer_Type", "Gene_A", "mech_A", "Gene_B", "mech_B", "n_elig", "n_both", "odds_ratio", "q_value"]
].to_string(index=False))
print("\nStrongest co-occurrence (cross-mechanism):")
print(cross[cross["odds_ratio"] > 1].nlargest(8, "odds_ratio")[
    ["Cancer_Type", "Gene_A", "mech_A", "Gene_B", "mech_B", "n_elig", "n_both", "odds_ratio", "q_value"]
].to_string(index=False))

significant cross-mechanism results: 374

Strongest mutual exclusivity (cross-mechanism):
               Cancer_Type Gene_A mech_A Gene_B mech_B  n_elig  n_both  odds_ratio       q_value
            Bladder Cancer   MDM2    CNV   TP53    SNV    5408      26    0.051206 5.164151e-101
Non-Small Cell Lung Cancer   EGFR    CNV   KRAS    SNV   22517      28    0.058117 5.449898e-128
             Breast Cancer   AKT1    SNV  ERBB2    CNV   16924       7    0.066368  1.020672e-30
            Bladder Cancer CDKN2B    CNV    RB1    SNV    5418      19    0.074954  2.311948e-60
            Bladder Cancer CDKN2A    CNV    RB1    SNV    5418      20    0.075243  1.190853e-62
Non-Small Cell Lung Cancer   KRAS    SNV PIK3CA    CNV   22519       9    0.077571  2.176299e-29
            Bladder Cancer  FGF19    CNV    RB1    SNV    4419       9    0.099486  2.637195e-21
            Bladder Cancer  CCND1    CNV    RB1    SNV    5371      12    0.102314  2.537149e-27

Strongest co-occurrence (cross-mecha

**These validate against known biology, which is the real test:**

- **`MDM2` amplification vs `TP53` mutation, mutually exclusive (OR=0.05)** --
  textbook. MDM2 amplification already disables p53 protein, so there is no
  selection pressure to also mutate `TP53`. Detectable *only* by comparing one
  gene's CNV against the other's SNV.
- **`EGFR` amplification vs `KRAS` mutation, exclusive (OR=0.06)** -- the
  classic lung-cancer driver exclusivity, holding across mechanisms.
- **`CDKN2A`/`CDKN2B` deletion vs `RB1` mutation, exclusive (OR=0.075)** --
  same p16-CDK4/6-RB pathway; once the pathway is broken one way, breaking it
  again adds nothing.

The co-occurrence side needs more caution: `PIK3CA` amplification with `TP53`
mutation in NSCLC (OR=13.5) likely reflects squamous histology, where 3q26
amplification and TP53 mutation are both common, rather than a direct
interaction. Cross-mechanism co-occurrence hits should be checked for subtype
confounding before being called biology.

### 5b. Does the driver-consistency filter earn its place?

Testing it on three pairs -- one suspected amplicon-passenger artifact and two
known-real findings.

In [9]:
def look_up(df, a, b, ct):
    m = (((df["Gene_A"] == a) & (df["Gene_B"] == b)) | ((df["Gene_A"] == b) & (df["Gene_B"] == a))) & df["Cancer_Type"].eq(ct)
    return df.loc[m, ["Gene_A", "mech_A", "Gene_B", "mech_B", "n_both", "odds_ratio", "q_value"]]

for a, b, ct in [("BRIP1", "GATA3", "Breast Cancer"),
                 ("MDM2", "TP53", "Bladder Cancer"),
                 ("EGFR", "KRAS", "Non-Small Cell Lung Cancer")]:
    print(f"=== {a} x {b} -- {ct}   [{a}: {GENE_ROLE.get(a)}, {b}: {GENE_ROLE.get(b)}]")
    print("  all deep CNVs:")
    print(look_up(mech_all, a, b, ct).to_string(index=False))
    print("  driver-consistent only:")
    d = look_up(mech_drv, a, b, ct)
    print(d.to_string(index=False) if not d.empty else "    (no driver-consistent CNV events remain)")
    print()

=== BRIP1 x GATA3 -- Breast Cancer   [BRIP1: TSG, GATA3: ONCOGENE_AND_TSG]
  all deep CNVs:
Gene_A mech_A Gene_B mech_B  n_both  odds_ratio       q_value
 BRIP1    SNV  GATA3    SNV      16    1.083754  8.492527e-01
 BRIP1    CNV  GATA3    SNV     436    8.222164 5.347040e-164
 BRIP1    CNV  GATA3    CNV      27    1.304231  2.627213e-01
  driver-consistent only:
Gene_A mech_A Gene_B mech_B  n_both  odds_ratio  q_value
 BRIP1    SNV  GATA3    SNV      16    1.083754 0.849253

=== MDM2 x TP53 -- Bladder Cancer   [MDM2: ONCOGENE, TP53: TSG]
  all deep CNVs:
Gene_A mech_A Gene_B mech_B  n_both  odds_ratio       q_value
  MDM2    SNV   TP53    SNV      16    1.175621  7.638619e-01
  MDM2    CNV   TP53    SNV      26    0.051206 5.164151e-101
  driver-consistent only:
Gene_A mech_A Gene_B mech_B  n_both  odds_ratio       q_value
  MDM2    SNV   TP53    SNV      16    1.175621  7.638619e-01
  MDM2    CNV   TP53    SNV      26    0.051206 4.186029e-101

=== EGFR x KRAS -- Non-Small Cell Lung 

**Clean result.** `BRIP1`(CNV) x `GATA3`(SNV) looked like a very strong
hit (OR=8.2, q=5e-164) -- but `BRIP1` is a **TSG** and the CNV involved was an
*amplification*, i.e. direction-inconsistent, almost certainly a passenger on
the 17q amplicon that is common in luminal breast cancer. Under the driver
filter **the hit disappears entirely**.

Meanwhile `MDM2`x`TP53` (oncogene + amplification = consistent) and
`EGFR`x`KRAS` survive essentially unchanged. The filter removes the artifact
class it was designed for without touching real biology.

### 5c. Pairs where the mechanism combos disagree in direction

The strongest argument for not blending mechanisms: pairs that are
significantly **co-occurring** under one mechanism combination and
significantly **mutually exclusive** under another. A single blended matrix
averages these into one muddled number.

In [10]:
s = mech_all[mech_all["q_value"] < 0.05].copy()
s["pair"] = [tuple(sorted([x, y])) for x, y in zip(s["Gene_A"], s["Gene_B"])]
conflicts = [(ct, pr, g) for (ct, pr), g in s.groupby(["Cancer_Type", "pair"])
             if g["odds_ratio"].max() > 1 and g["odds_ratio"].min() < 1]
print(f"{len(conflicts)} pairs significant in BOTH directions depending on mechanism "
      f"(of {s['pair'].nunique()} significant pairs)\n")
for ct, pr, g in conflicts[:10]:
    detail = " | ".join(f"{r.combo} OR={r.odds_ratio:.2f}" for r in g.itertuples())
    print(f"  {ct:28s} {pr[0]}-{pr[1]:8s} {detail}")

88 pairs significant in BOTH directions depending on mechanism (of 1729 significant pairs)

  Bladder Cancer               ASXL1-KDM6A    SNV-SNV OR=1.60 | CNV-SNV OR=0.35
  Bladder Cancer               CCND1-ERBB3    CNV-SNV OR=0.48 | CNV-CNV OR=4.78
  Bladder Cancer               CCND1-HRAS     CNV-SNV OR=0.24 | CNV-CNV OR=7.88
  Bladder Cancer               CCND1-PIK3CA   SNV-SNV OR=3.48 | CNV-SNV OR=0.69
  Bladder Cancer               CDKN2A-CDKN2B   SNV-CNV OR=0.18 | CNV-CNV OR=6454.77
  Bladder Cancer               CDKN2A-TP53     SNV-SNV OR=1.77 | CNV-SNV OR=0.44 | CNV-CNV OR=4.30
  Bladder Cancer               CDKN2B-TP53     CNV-SNV OR=0.44 | CNV-CNV OR=2.50
  Bladder Cancer               CREBBP-FGFR1    SNV-SNV OR=3.08 | SNV-CNV OR=0.33
  Bladder Cancer               DDR2-PIK3CA   SNV-SNV OR=1.80 | CNV-SNV OR=0.55
  Bladder Cancer               ERBB2-PIK3CA   SNV-SNV OR=1.27 | CNV-SNV OR=0.45


`CCND1`-`HRAS` in bladder cancer is a clear example: their **amplifications**
strongly co-occur (OR=7.9, both are on amplicons that travel together) while
`CCND1` amplification is mutually exclusive with `HRAS` **mutation** (OR=0.24,
two alternative routes to the same pathway). Those are two genuinely different
biological statements about the same gene pair, and the current blended matrix
reports neither.

(`CDKN2A`-`CDKN2B` with CNV-CNV OR=6455 is the known physical-adjacency
artifact -- adjacent genes at 9p21 always deleted in the same event. Notebook
05's genomic-proximity flag already handles that class.)

### 5d. Subtype confounding check on the cross-mechanism hits

Section 5a flagged that cross-mechanism *co-occurrence* hits could be driven by
histology rather than by any direct interaction. That is testable directly:
stratify by `CANCER_TYPE_DETAILED` and see whether the association survives
inside each histological subtype.

Two pairs, one suspected confounded and one expected to be real:

In [11]:
def stratified_check(cancer_type, gene_a, mech_a, gene_b, mech_b, by="CANCER_TYPE_DETAILED", min_alt=10):
    cs = samples_full[(samples_full["CANCER_TYPE"] == cancer_type)
                      & samples_full["SEQ_ASSAY_ID"].isin(CNV_OK_PANELS)].set_index("SAMPLE_ID")
    cov = panel_coverage[panel_coverage["Hugo_Symbol"].isin([gene_a, gene_b])].groupby("SEQ_ASSAY_ID")["Hugo_Symbol"].apply(set)
    cs = cs[cs["SEQ_ASSAY_ID"].isin(cov[cov.apply(lambda x: {gene_a, gene_b} <= x)].index)]

    def vec(gene, mech):
        t = "MUT" if mech == "SNV" else "CNV"
        ids = set(alterations.loc[(alterations["Hugo_Symbol"] == gene) & (alterations["Alteration_Type"] == t), "Sample_ID"])
        return cs.index.isin(ids)

    A, B = vec(gene_a, mech_a), vec(gene_b, mech_b)

    def cell(a, b):
        both, ao = int((a & b).sum()), int((a & ~b).sum())
        bo, ne = int((~a & b).sum()), int((~a & ~b).sum())
        orv, pv = fisher_exact([[both, ao], [bo, ne]])
        return both, orv, pv

    both, orv, pv = cell(A, B)
    rows = [dict(stratum="POOLED (all subtypes)", n=len(cs), n_both=both, odds_ratio=round(orv, 2), p_value=pv)]
    for lev, idx in cs.groupby(by).groups.items():
        m = cs.index.isin(idx)
        a, b = A[m], B[m]
        if int((a).sum()) < min_alt or int((b).sum()) < min_alt:
            continue
        both, orv, pv = cell(a, b)
        rows.append(dict(stratum=str(lev)[:45], n=int(m.sum()), n_both=both, odds_ratio=round(orv, 2), p_value=pv))
    return pd.DataFrame(rows)


samples_full = clinical[["SAMPLE_ID", "SEQ_ASSAY_ID", "CANCER_TYPE", "CANCER_TYPE_DETAILED"]].drop_duplicates("SAMPLE_ID")

print("SUSPECTED CONFOUNDED -- PIK3CA(CNV) x TP53(SNV), NSCLC")
print(stratified_check("Non-Small Cell Lung Cancer", "PIK3CA", "CNV", "TP53", "SNV").head(6).to_string(index=False))
print()
print("EXPECTED REAL -- MDM2(CNV) x TP53(SNV), Bladder Cancer")
print(stratified_check("Bladder Cancer", "MDM2", "CNV", "TP53", "SNV").head(5).to_string(index=False))

SUSPECTED CONFOUNDED -- PIK3CA(CNV) x TP53(SNV), NSCLC


                     stratum     n  n_both  odds_ratio      p_value
       POOLED (all subtypes) 22519     295       13.46 8.953860e-62
         Lung Adenocarcinoma 17163      26        3.45 9.403713e-04
Lung Squamous Cell Carcinoma  2151     232        6.64 8.834658e-12
  Non-Small Cell Lung Cancer  1689      29        8.07 4.022503e-04

EXPECTED REAL -- MDM2(CNV) x TP53(SNV), Bladder Cancer


                         stratum    n  n_both  odds_ratio       p_value
           POOLED (all subtypes) 5408      26        0.05 8.289167e-104
    Bladder Urothelial Carcinoma 4168      19        0.04  1.077054e-88
Upper Tract Urothelial Carcinoma  797       3        0.06  1.968200e-12


**The check separates the two cleanly.**

`PIK3CA`(CNV) x `TP53`(SNV) in NSCLC: pooled **OR=13.5**, but inside each
histology it drops to **3.45** (adenocarcinoma), **6.64** (squamous), **8.07**
(NSCLC NOS). Squamous supplies 232 of the 295 co-occurring samples while being
only ~2,150 of 22,519 -- so histology inflates the pooled estimate roughly
2-4x. This is textbook confounding (Simpson's-paradox shaped). Note it does
*not* vanish -- the association stays significant in every stratum, so there is
a real residual signal; the pooled number just badly overstates it.

`MDM2`(CNV) x `TP53`(SNV) in bladder cancer: pooled **OR=0.05**, stratified
**0.04** and **0.06** -- essentially unchanged. Real biology, no confounding.

**Implication:** cross-mechanism hits are worth reporting, but co-occurrence
hits need this subtype check before being described as biology. Mutual
exclusivity hits look far more robust in this pilot.

**And a useful contrast with the panel batch-effect question from notebook 05:**
stratification is exactly the right tool *here* -- histology strata are few and
large (17,163 / 2,151 / 1,689 samples), so the stratified estimates are stable.
That is precisely the condition the 167 sequencing panels fail. Same technique,
opposite verdict, for a concrete and checkable reason.

## 6. Saving the state annotation

In [12]:
out = states[["Sample_ID", "Hugo_Symbol", "state", "has_snv", "has_cnv", "CNA_Direction", "Gene_Role"]].copy()
out.to_parquet(PROCESSED_DIR / "gene_alteration_states.parquet", index=False)
print(f"wrote gene_alteration_states.parquet: {len(out):,} rows")
print("NOTE: restricted to samples on CNV-reporting panels -- a state-1 row here means "
      "'SNV only, and this sample COULD have shown a CNV', which is the only way state 1 vs 3 is meaningful.")
out.head()

wrote gene_alteration_states.parquet: 1,216,243 rows
NOTE: restricted to samples on CNV-reporting panels -- a state-1 row here means 'SNV only, and this sample COULD have shown a CNV', which is the only way state 1 vs 3 is meaningful.


,Sample_ID,Hugo_Symbol,state,has_snv,has_cnv,CNA_Direction,Gene_Role
0,GENIE-CRUK-MB0002-primary,FOXO3,1,True,False,NaN,INSUFFICIENT_EVIDENCE
1,GENIE-CRUK-MB0002-primary,MEN1,2,False,True,Amplification,TSG
2,GENIE-CRUK-MB0002-primary,NF1,2,False,True,Deep_Deletion,TSG
3,GENIE-CRUK-MB0002-primary,SBNO1,2,False,True,Deep_Deletion,NaN
4,GENIE-CRUK-MB0002-primary,TBL1XR1,2,False,True,Deep_Deletion,ONCOGENE


## Discussion

**On the state model (your proposal).** Worth building, and now built. But the
framing needs one correction: **state 3 is not a two-hit marker.** 58.6% of
state-3 events are oncogene + amplification (two drivers stacking, not
inactivation); only 22.2% are TSG + deletion; and even those are a *different*
event class from the Knudson two-hit Phase 1 already models with shallow
deletions -- zero overlap between the two sets. State 3 is best described as
"this gene was hit two different ways", which is genuinely interesting
(`EGFR` mutation + amplification, ~1,650 events, is a real dual-driver
pattern) but should not be labelled two-hit in the writeup.

**On the 4x4 categorical test.** Your instinct to flag sparsity was right, and
the data is worse than expected: state 3 is only 0.7% of altered gene-sample
pairs, so even the most favourable pair available (`EGFR` x `TP53` in the
largest cancer type) has state-3 cells of 1-5 samples. I'd drop the 4x4
omnibus test rather than carry it as a caveated supplementary -- it also
returns only an undirected p-value, which is strictly less useful than what
follows.

**On mechanism-specific 2x2 tests (recommended instead).** This keeps
everything you wanted from the categorical model -- seeing *which* mechanism
combination carries a pair's signal -- while staying on the robust 2x2 Fisher
machinery. It earns its place on evidence, not theory:
- it recovers `MDM2`-amp vs `TP53`-mut exclusivity, which **no** single-mechanism
  matrix can see;
- 88 pairs flip direction depending on mechanism, meaning the current blended
  matrix is actively averaging away real, opposite signals;
- combined with the driver filter it removed a q=5e-164 artifact (`BRIP1`
  amplicon passenger) while leaving textbook results untouched.

**Suggested final shape** (differs from the plan we discussed, on evidence):

| layer | what | status |
|---|---|---|
| main | Combined 2x2 matrix, as now | keep |
| main | SNV-only and CNV-only 2x2 matrices | add (notebook 05) |
| main | mechanism-specific SNVxCNV cross tests | **add -- this is where the unique findings are** |
| annotation | driver-consistency on CNV calls | add to Phase 1 |
| annotation | per-gene state 0/1/2/3 | add, but describe honestly |
| annotation | two-hit (shallow del + mut) | already exists, keep separate |
| dropped | 4x4 categorical omnibus test | too sparse to support |

**Blocking prerequisite:** all of this depends on the `cna_reports_calls` fix
from notebook 05. Until CNV-blind panels are excluded, every CNV-involving
state and test is computed against a denominator that is ~34% samples which
could never have produced a positive call.

**For tomorrow / for Jason:** the open calls are (1) whether to report state 3
at all given the two-hit framing doesn't hold, (2) the driver-filter strictness
(strict `Driver_Consistent` vs including `Unannotated`), (3) whether
cross-mechanism co-occurrence hits need a histology/subtype control before
being reported as biology.